In [ ]:
%pip install -U langgraph
%pip install -U langchain

In [2]:
import os
from langchain.chat_models import init_chat_model


# Practising with Lang Graph

## 1. Making tools

In [25]:
from langchain.tools import tool
import requests

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [20]:
model = init_chat_model(
    "gpt-4.1",
    temperature=0
    )

In [78]:
@tool
def addTool(a: int, b: int) -> int:
    """Adds `a` and `b`.

    Args:
        a: First int
        b: Second int
    """
    return a + b


@tool
def weatherTool(lat: float, lon: float) -> str:
    """
    Returns the current weather in a location given a lat, lon
    """
    url = (
        f"https://api.open-meteo.com/v1/forecast?"
        f"latitude={lat}&longitude={lon}"
        "&daily=temperature_2m_max,temperature_2m_min,relative_humidity_2m_mean"
        "&timezone=auto"
    )
    res = requests.get(url).json()
    return res

In [91]:
math_tools = [addTool]
math_tools_by_name = {t.name: t for t in math_tools}
model_with_math_tools = model.bind_tools(math_tools)

In [92]:
weather_tools = [weatherTool]
weather_tools_by_name = {t.name: t for t in weather_tools}
model_with_weather_tools = model.bind_tools(weather_tools)

In [23]:
lat = 1.3521
lon = 103.8198
url = (
    f"https://api.open-meteo.com/v1/forecast?"
    f"latitude={lat}&longitude={lon}"
    "&daily=temperature_2m_max,temperature_2m_min,relative_humidity_2m_mean"
    "&timezone=auto"
)
res = requests.get(url).json()

In [ ]:
res

In [ ]:
print(res['daily']['temperature_2m_max'][0])

## 2. Build graph

In [135]:
from typing import Annotated, List
from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel
from langchain_core.messages import HumanMessage, AIMessage

class OverallState(BaseModel):
    instruction:    str
    intent:         str | None = None
    answer:         str | None = None
    approved:       bool | None = None
    messages:       Annotated[List[AnyMessage], add_messages] = []
    model_config = {"arbitrary_types_allowed": True}

In [103]:
def classify_intent_node(state: OverallState):
    prompt = f"""You are an intent classifier.
    User instruction: "{state.instruction}"

    Possible intents: ["math", "weather"]
    Respond with only the intent label.
    """
    response = model.invoke(prompt)
    intent = response.content.strip().lower()
    return {"intent": intent}

def route_from_intent(state: OverallState):
    if state.intent == "math":
        return "addition_node"
    elif state.intent == "weather":
        return "weather_node"
    else:
        return END

In [137]:
def addition_node(state: OverallState):
    # If no messages yet, seed them
    if not state.messages:
        state.messages.append(HumanMessage(content=state.instruction))

    # Call the LLM with tools using the message history
    response = model_with_math_tools.invoke(state.messages)

    # Append the new AI message to messages
    state.messages.append(response)

    # If the AI emitted tool calls, run them
    if getattr(response, "tool_calls", None):
        results = []
        for tc in response.tool_calls:
            tool = math_tools_by_name[tc["name"]]
            result = tool.invoke(tc["args"])
            results.append(f"{tc['name']} result: {result}")
        return {
            "answer": "\n".join(results),
            "messages": state.messages,  # optional (add_messages will merge)
        }

    # Otherwise just return the text
    return {
        "answer": response.content,
        "messages": state.messages,
    }
    
def weather_node(state: OverallState):
    # 1. Build the messages we send to the model (fresh each time)
    call_messages = [HumanMessage(content=state.instruction)]

    # 2. Call the tool-enabled model
    response = model_with_weather_tools.invoke(call_messages)

    # 3. If there are tool calls, run them and build the final answer text
    if getattr(response, "tool_calls", None):
        results = []
        for tc in response.tool_calls:
            tool = weather_tools_by_name[tc["name"]]
            result = tool.invoke(tc["args"])
            results.append(str(result))
        answer_text = "\n".join(results)
    else:
        answer_text = response.content

    # 4. Maintain a "clean" message history in state.messages (no tool_calls)
    messages = state.messages or []
    messages.append(HumanMessage(content=state.instruction))
    messages.append(AIMessage(content=answer_text))

    return {
        "answer": answer_text,
        "messages": messages,
    }

In [162]:
def checker_node(state: OverallState):
    prompt = f"""
You are an answer checker.

User instruction:
{state.instruction}

Proposed answer:
{state.answer}

Decide if the answer is acceptable. For weather forecasts, JSON data is acceptable.
Reply with a verbose answer that must begin with: "approved" or "reject".
"""
    response = model.invoke(prompt)
    full_verdict_text = response.content.strip()
    lower_verdict = full_verdict_text.lower()

    approved = lower_verdict.startswith("approved")

    # Use existing message history, or start fresh
    messages = state.messages or []

    # Log the checker result as an AI message (tagged if you like)
    messages.append(
        AIMessage(content=f"[CHECKER]\n{full_verdict_text}")
    )

    return {
        "approved": approved,
        "messages": messages,
    }

def route_from_checker(state: OverallState):
    # example: if approved, finish; else, re-run the pipeline
    if state.approved:
        return END
    else:
        return "classify_intent_node"


In [ ]:
builder = StateGraph(OverallState)

builder.add_node("classify_intent_node", classify_intent_node)
builder.add_node("addition_node", addition_node)
builder.add_node("weather_node", weather_node)
builder.add_node("checker_node", checker_node)

In [164]:
builder.add_edge(START, "classify_intent_node")

builder.add_conditional_edges(
    "classify_intent_node",
    route_from_intent,
    ["addition_node", "weather_node", END]
)

builder.add_edge("addition_node", "checker_node")
builder.add_edge("weather_node", "checker_node")

builder.add_conditional_edges(
    "checker_node",
    route_from_checker,
    ["classify_intent_node", END],  # ✅ include "classify_intent" here
)

graph = builder.compile()

In [ ]:
result = graph.invoke(
    {
        "instruction": "2+2"
    }
)
result

In [ ]:
from IPython.display import Image, display
display(Image(graph.get_graph(xray=True).draw_mermaid_png()))

# Quick Start

In [3]:
from langchain.tools import tool

In [4]:
model = init_chat_model(
    "gpt-4.1",
    temperature=0
    )

In [5]:
# Define tools
@tool
def multiplyTool(a: int, b: int) -> int:
    """Multiply `a` and `b`.

    Args:
        a: First int
        b: Second int
    """
    return a * b


@tool
def addTool(a: int, b: int) -> int:
    """Adds `a` and `b`.

    Args:
        a: First int
        b: Second int
    """
    return a + b


@tool
def divideTool(a: int, b: int) -> float:
    """Divide `a` and `b`.

    Args:
        a: First int
        b: Second int
    """
    return a / b

In [8]:
tools = [addTool, multiplyTool, divideTool]
tools_by_name = {tool.name: tool for tool in tools}
model_with_tools = model.bind_tools(tools)

In [9]:
from langchain.messages import AnyMessage
from typing_extensions import TypedDict, Annotated
import operator


class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]
    llm_calls: int

In [10]:
from langchain.messages import SystemMessage


def llm_call(state: dict):
    """LLM decides whether to call a tool or not"""

    return {
        "messages": [
            model_with_tools.invoke(
                [
                    SystemMessage(
                        content="You are a helpful assistant tasked with performing arithmetic on a set of inputs."
                    )
                ]
                + state["messages"]
            )
        ],
        "llm_calls": state.get('llm_calls', 0) + 1
    }

In [11]:
from langchain.messages import ToolMessage


def tool_node(state: dict):
    """Performs the tool call"""

    result = []
    for tool_call in state["messages"][-1].tool_calls:
        tool = tools_by_name[tool_call["name"]]
        observation = tool.invoke(tool_call["args"])
        result.append(ToolMessage(content=observation, tool_call_id=tool_call["id"]))
    return {"messages": result}

In [12]:
from typing import Literal
from langgraph.graph import StateGraph, START, END


def should_continue(state: MessagesState) -> Literal["tool_node", END]:
    """Decide if we should continue the loop or stop based upon whether the LLM made a tool call"""

    messages = state["messages"]
    last_message = messages[-1]

    # If the LLM makes a tool call, then perform an action
    if last_message.tool_calls:
        return "tool_node"

    # Otherwise, we stop (reply to the user)
    return END

In [ ]:
# Build workflow
agent_builder = StateGraph(MessagesState)

# Add nodes
agent_builder.add_node("llm_call", llm_call)
agent_builder.add_node("tool_node", tool_node)

# Add edges to connect nodes
agent_builder.add_edge(START, "llm_call")
agent_builder.add_conditional_edges(
    "llm_call",
    should_continue,
    ["tool_node", END]
)
agent_builder.add_edge("tool_node", "llm_call")

# Compile the agent
agent = agent_builder.compile()

# Show the agent
from IPython.display import Image, display
display(Image(agent.get_graph(xray=True).draw_mermaid_png()))

# Invoke
from langchain.messages import HumanMessage
messages = [HumanMessage(content="Add 3 and 4.")]
messages = agent.invoke({"messages": messages})
for m in messages["messages"]:
    print("Printing a message")
    m.pretty_print()
    print("Ending message")